# ETL con archivos de texto

## Selección Argentina - Mundial 2026

En este notebook vamos a trabajar con tres archivos de texto semi-estructurados:

- `data/mundial/seleccion_argentina_mundial_2026.txt`
- `data/mundial/seleccion_argentina_mundial_2026_clubes_continentes.txt`
- `data/mundial/seleccion_argentina_mundial_2026_edades.txt`

La idea es recorrer un proceso ETL simple:

1. **Extraer** datos desde archivos `.txt`.
2. **Transformar** texto libre en tablas.
3. **Integrar** las tablas en un único `DataFrame`.
4. **Analizar** el plantel con agrupaciones y preguntas disparadoras.

## Pregunta inicial

> Si tenemos tres archivos de texto con información parcial sobre los mismos jugadores, ¿cómo hacemos para construir una única tabla útil para analizar?

## Librerías

In [1]:
from pathlib import Path

import pandas as pd

## Extract: leer los archivos crudos

Primero definimos las rutas de entrada. El notebook asume que se ejecuta desde la carpeta `etl-01`.

In [2]:
data_dir = Path("data/mundial")

archivo_posiciones = data_dir / "seleccion_argentina_mundial_2026.txt"
archivo_clubes = data_dir / "seleccion_argentina_mundial_2026_clubes_continentes.txt"
archivo_edades = data_dir / "seleccion_argentina_mundial_2026_edades.txt"

archivos = [archivo_posiciones, archivo_clubes, archivo_edades]

for archivo in archivos:
    print(archivo, "->", archivo.exists())

data/mundial/seleccion_argentina_mundial_2026.txt -> True
data/mundial/seleccion_argentina_mundial_2026_clubes_continentes.txt -> True
data/mundial/seleccion_argentina_mundial_2026_edades.txt -> True


Leemos el contenido como texto. En esta etapa todavía no tenemos una tabla: tenemos líneas.

In [3]:
lineas_posiciones = archivo_posiciones.read_text(encoding="utf-8").splitlines()
lineas_clubes = archivo_clubes.read_text(encoding="utf-8").splitlines()
lineas_edades = archivo_edades.read_text(encoding="utf-8").splitlines()

lineas_posiciones[:12]

['Seleccion Argentina - Copa Mundial FIFA 2026',
 'Actualizado: 9 de junio de 2026',
 '',
 'Nota: la AFA anuncio la lista final de 26 jugadores el 28 de mayo de 2026.',
 'Leonardo Balerdi fue baja por lesion el 6 de junio de 2026 y, al momento de',
 'esta actualizacion, su reemplazo aun no estaba confirmado.',
 '',
 'Jugadores y posiciones',
 '',
 '- Juan Musso - Arquero',
 '- Geronimo Rulli - Arquero',
 '- Emiliano Martinez - Arquero']

## Transform: quedarnos solo con las líneas de jugadores

Los archivos tienen títulos, notas y una sección especial para bajas. Para el análisis principal queremos solo la sección `Jugadores`.

In [4]:
def extraer_lineas_jugadores(lineas):
    en_jugadores = False
    resultado = []

    for linea in lineas:
        linea = linea.strip()
        if "Jugadores" in linea:
            en_jugadores = True
            continue

        if linea == "Baja pendiente de reemplazo":
            en_jugadores = False
            continue
        

        if en_jugadores and linea.startswith("- "):
            resultado.append(linea.removeprefix("- "))

    return resultado



In [5]:
lineas_posiciones[:12]

['Seleccion Argentina - Copa Mundial FIFA 2026',
 'Actualizado: 9 de junio de 2026',
 '',
 'Nota: la AFA anuncio la lista final de 26 jugadores el 28 de mayo de 2026.',
 'Leonardo Balerdi fue baja por lesion el 6 de junio de 2026 y, al momento de',
 'esta actualizacion, su reemplazo aun no estaba confirmado.',
 '',
 'Jugadores y posiciones',
 '',
 '- Juan Musso - Arquero',
 '- Geronimo Rulli - Arquero',
 '- Emiliano Martinez - Arquero']

In [6]:
jugadores_posiciones = extraer_lineas_jugadores(lineas_posiciones)
jugadores_posiciones[:5]

['Juan Musso - Arquero',
 'Geronimo Rulli - Arquero',
 'Emiliano Martinez - Arquero',
 'Nicolas Tagliafico - Defensor',
 'Gonzalo Montiel - Defensor']

In [7]:

jugadores_posiciones = extraer_lineas_jugadores(lineas_posiciones)
jugadores_clubes = extraer_lineas_jugadores(lineas_clubes)
jugadores_edades = extraer_lineas_jugadores(lineas_edades)

print(jugadores_posiciones)
print(jugadores_clubes)
print(jugadores_edades)

['Juan Musso - Arquero', 'Geronimo Rulli - Arquero', 'Emiliano Martinez - Arquero', 'Nicolas Tagliafico - Defensor', 'Gonzalo Montiel - Defensor', 'Lisandro Martinez - Defensor', 'Cristian Romero - Defensor', 'Nicolas Otamendi - Defensor', 'Facundo Medina - Defensor', 'Nahuel Molina - Defensor', 'Leandro Paredes - Mediocampista', 'Rodrigo De Paul - Mediocampista', 'Valentin Barco - Mediocampista', 'Giovani Lo Celso - Mediocampista', 'Exequiel Palacios - Mediocampista', 'Nicolas Gonzalez - Mediocampista', 'Alexis Mac Allister - Mediocampista', 'Enzo Fernandez - Mediocampista', 'Julian Alvarez - Delantero', 'Lionel Messi - Delantero', 'Thiago Almada - Delantero', 'Giuliano Simeone - Delantero', 'Nico Paz - Delantero', 'Jose Manuel Lopez - Delantero', 'Lautaro Martinez - Delantero']
['Juan Musso - Atletico Madrid - Europa', 'Geronimo Rulli - Marseille - Europa', 'Emiliano Martinez - Aston Villa - Europa', 'Nicolas Tagliafico - Lyon - Europa', 'Gonzalo Montiel - River Plate - America del S

### Pregunta

> ¿Por qué no conviene leer todas las líneas que empiezan con `- ` sin mirar la sección donde aparecen?

## Transform: convertir texto en tablas

Cada archivo usa un separador simple: ` - `.

In [8]:
def tabla_posiciones(lineas_jugadores):
    filas = []

    for linea in lineas_jugadores:
        jugador, posicion = linea.split(" - ")
        filas.append({"jugador": jugador, "posicion": posicion})

    return pd.DataFrame(filas)


def tabla_clubes(lineas_jugadores):
    filas = []

    for linea in lineas_jugadores:
        jugador, club, continente = linea.split(" - ")
        filas.append({"jugador": jugador, "club": club, "continente": continente})

    return pd.DataFrame(filas)


def tabla_edades(lineas_jugadores):
    filas = []

    for linea in lineas_jugadores:
        jugador, edad = linea.split(" - ")
        filas.append({"jugador": jugador, "edad": int(edad)})

    return pd.DataFrame(filas)

In [9]:
df_posiciones = tabla_posiciones(jugadores_posiciones)
df_clubes = tabla_clubes(jugadores_clubes)
df_edades = tabla_edades(jugadores_edades)

df_posiciones.head()

,jugador,posicion
0,Juan Musso,Arquero
1,Geronimo Rulli,Arquero
2,Emiliano Martinez,Arquero
3,Nicolas Tagliafico,Defensor
4,Gonzalo Montiel,Defensor


In [10]:
df_clubes.head()

,jugador,club,continente
0,Juan Musso,Atletico Madrid,Europa
1,Geronimo Rulli,Marseille,Europa
2,Emiliano Martinez,Aston Villa,Europa
3,Nicolas Tagliafico,Lyon,Europa
4,Gonzalo Montiel,River Plate,America del Sur


In [11]:
df_edades.head()

,jugador,edad
0,Juan Musso,32
1,Geronimo Rulli,34
2,Emiliano Martinez,33
3,Nicolas Tagliafico,33
4,Gonzalo Montiel,29


## Validación inicial

Antes de combinar tablas, revisamos si tienen la misma cantidad de jugadores y si aparecen duplicados.

In [12]:
resumen_fuentes = pd.DataFrame({
    "fuente": ["posiciones", "clubes_continentes", "edades"],
    "filas": [len(df_posiciones), len(df_clubes), len(df_edades)],
    "jugadores_unicos": [
        df_posiciones["jugador"].nunique(),
        df_clubes["jugador"].nunique(),
        df_edades["jugador"].nunique(),
    ],
    "duplicados": [
        df_posiciones["jugador"].duplicated().sum(),
        df_clubes["jugador"].duplicated().sum(),
        df_edades["jugador"].duplicated().sum(),
    ],
})

resumen_fuentes

,fuente,filas,jugadores_unicos,duplicados
0,posiciones,25,25,0
1,clubes_continentes,25,25,0
2,edades,25,25,0


### Pregunta

> ¿Qué problema tendríamos si un jugador aparece escrito distinto en dos archivos?

## Load: integrar en una única tabla

Una forma simple de hace resto es:

In [12]:
df = (
    df_posiciones.set_index("jugador")
    .join(df_clubes.set_index("jugador"))
    .join(df_edades.set_index("jugador"))
    .reset_index()
)
df.head()

,jugador,posicion,club,continente,edad
0,Juan Musso,Arquero,Atletico Madrid,Europa,32
1,Geronimo Rulli,Arquero,Marseille,Europa,34
2,Emiliano Martinez,Arquero,Aston Villa,Europa,33
3,Nicolas Tagliafico,Defensor,Lyon,Europa,33
4,Gonzalo Montiel,Defensor,River Plate,America del Sur,29


Usamos `merge` para unir las tablas por la columna `jugador`.

In [14]:
df = (
    df_posiciones
    .merge(df_clubes, on="jugador", how="inner")
    .merge(df_edades, on="jugador", how="inner")
)

df.head()

,jugador,posicion,club,continente,edad
0,Juan Musso,Arquero,Atletico Madrid,Europa,32
1,Geronimo Rulli,Arquero,Marseille,Europa,34
2,Emiliano Martinez,Arquero,Aston Villa,Europa,33
3,Nicolas Tagliafico,Defensor,Lyon,Europa,33
4,Gonzalo Montiel,Defensor,River Plate,America del Sur,29


In [13]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 25 entries, 0 to 24
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   jugador     25 non-null     str  
 1   posicion    25 non-null     str  
 2   club        25 non-null     str  
 3   continente  25 non-null     str  
 4   edad        25 non-null     int64
dtypes: int64(1), str(4)
memory usage: 2.2 KB


### Pregunta

> ¿Por qué usamos `jugador` como clave de unión? ¿Qué pasaría si hubiera dos jugadores con el mismo nombre?

## Dataset final

In [14]:
df.sort_values("jugador")

,jugador,posicion,club,continente,edad
16,Alexis Mac Allister,Mediocampista,Liverpool,Europa,27
6,Cristian Romero,Defensor,Tottenham Hotspur,Europa,28
2,Emiliano Martinez,Arquero,Aston Villa,Europa,33
17,Enzo Fernandez,Mediocampista,Chelsea,Europa,25
14,Exequiel Palacios,Mediocampista,Bayer Leverkusen,Europa,27
8,Facundo Medina,Defensor,Marseille,Europa,27
1,Geronimo Rulli,Arquero,Marseille,Europa,34
13,Giovani Lo Celso,Mediocampista,Real Betis,Europa,30
21,Giuliano Simeone,Delantero,Atletico Madrid,Europa,23
4,Gonzalo Montiel,Defensor,River Plate,America del Sur,29


## Análisis 1: jugadores por posición

> ¿Qué posiciones tienen más jugadores convocados?

In [17]:
(
    df
    .groupby("posicion")
    .size()    
)

posicion
Arquero          3
Defensor         7
Delantero        7
Mediocampista    8
dtype: int64

In [18]:
(
    df
    .groupby("posicion")
    .size()
    .reset_index(name="cantidad")
    .sort_values("cantidad", ascending=False)
)

,posicion,cantidad
3,Mediocampista,8
1,Defensor,7
2,Delantero,7
0,Arquero,3


> Usamos `( ... )` para que quede prolijo

## Análisis 2: jugadores por continente

> ¿En qué continente juegan más futbolistas del plantel?

In [19]:
(
    df
    .groupby("continente")
    .size()
    .reset_index(name="cantidad")
    .sort_values("cantidad", ascending=False)
)

,continente,cantidad
2,Europa,20
1,America del Sur,3
0,America del Norte,2


## Análisis 3: clubes con más jugadores

> ¿Qué clubes aportan más jugadores?

In [20]:
(
    df
    .groupby("club")
    .size()
    .reset_index(name="cantidad")
    .sort_values(["cantidad", "club"], ascending=[False, True])
)

,club,cantidad
1,Atletico Madrid,6
7,Inter Miami,2
12,Marseille,2
0,Aston Villa,1
2,Bayer Leverkusen,1
3,Benfica,1
4,Boca Juniors,1
5,Chelsea,1
6,Como,1
8,Inter Milan,1


## Análisis 4: edad promedio del plantel

> ¿Cuál es la edad promedio del plantel? ¿Quiénes están por encima de ese promedio?

In [15]:
edad_promedio = df["edad"].mean()
edad_promedio

np.float64(28.68)

In [16]:
(
    df
    .assign(mayor_que_promedio=lambda datos: datos["edad"] > edad_promedio)
    .sort_values("edad", ascending=False)
)

,jugador,posicion,club,continente,edad,mayor_que_promedio
19,Lionel Messi,Delantero,Inter Miami,America del Norte,38,True
7,Nicolas Otamendi,Defensor,Benfica,Europa,38,True
1,Geronimo Rulli,Arquero,Marseille,Europa,34,True
2,Emiliano Martinez,Arquero,Aston Villa,Europa,33,True
3,Nicolas Tagliafico,Defensor,Lyon,Europa,33,True
0,Juan Musso,Arquero,Atletico Madrid,Europa,32,True
11,Rodrigo De Paul,Mediocampista,Inter Miami,America del Norte,32,True
10,Leandro Paredes,Mediocampista,Boca Juniors,America del Sur,31,True
13,Giovani Lo Celso,Mediocampista,Real Betis,Europa,30,True
4,Gonzalo Montiel,Defensor,River Plate,America del Sur,29,True


## Análisis 5: edad por posición

> ¿Qué posición tiene el promedio de edad más alto? ¿Y el más bajo?

In [23]:
(
    df
    .groupby("posicion")
    .agg(
        cantidad=("jugador", "count"),
        edad_promedio=("edad", "mean"),
        edad_minima=("edad", "min"),
        edad_maxima=("edad", "max"),
    )
    .reset_index()
    .sort_values("edad_promedio", ascending=False)
)

,posicion,cantidad,edad_promedio,edad_minima,edad_maxima
0,Arquero,3,33.000000,32,34
1,Defensor,7,30.142857,27,38
3,Mediocampista,8,27.625000,21,32
2,Delantero,7,26.571429,21,38


## Análisis 6: posición y continente

> ¿Cómo se distribuyen las posiciones según el continente donde juegan?

In [24]:
pd.crosstab(df["posicion"], df["continente"])

continente,America del Norte,America del Sur,Europa
posicion,,,
Arquero,0,0,3
Defensor,0,1,6
Delantero,1,1,5
Mediocampista,1,1,6


## Análisis 7: jugadores jóvenes y veteranos

> Si definimos jugadores jóvenes como menores de 25 años y veteranos como mayores de 32, ¿quiénes entran en cada grupo?

In [18]:
df_clasificado = df.assign(
    grupo_edad=pd.cut(
        df["edad"],
        bins=[0, 24, 29, 100],
        labels=["joven", "intermedio", "veterano"],
    )
)

df_clasificado.sort_values(["grupo_edad", "edad", "jugador"])

,jugador,posicion,club,continente,edad,grupo_edad
22,Nico Paz,Delantero,Como,Europa,21,joven
12,Valentin Barco,Mediocampista,Strasbourg,Europa,21,joven
21,Giuliano Simeone,Delantero,Atletico Madrid,Europa,23,joven
17,Enzo Fernandez,Mediocampista,Chelsea,Europa,25,intermedio
23,Jose Manuel Lopez,Delantero,Palmeiras,America del Sur,25,intermedio
20,Thiago Almada,Delantero,Atletico Madrid,Europa,25,intermedio
18,Julian Alvarez,Delantero,Atletico Madrid,Europa,26,intermedio
16,Alexis Mac Allister,Mediocampista,Liverpool,Europa,27,intermedio
14,Exequiel Palacios,Mediocampista,Bayer Leverkusen,Europa,27,intermedio
8,Facundo Medina,Defensor,Marseille,Europa,27,intermedio


In [19]:
(
    df_clasificado
    .groupby("grupo_edad", observed=True)
    .size()
    .reset_index(name="cantidad")
)

,grupo_edad,cantidad
0,joven,3
1,intermedio,13
2,veterano,9


## Calidad de datos: la baja pendiente de reemplazo

Los archivos también tienen una sección `Baja pendiente de reemplazo`. Esa información no forma parte del plantel activo, pero puede ser relevante para auditoría.

In [27]:
def extraer_lineas_baja(lineas):
    en_baja = False
    resultado = []

    for linea in lineas:
        linea = linea.strip()

        if linea == "Baja pendiente de reemplazo":
            en_baja = True
            continue

        if en_baja and linea.startswith("- "):
            resultado.append(linea.removeprefix("- "))

    return resultado


bajas_posiciones = extraer_lineas_baja(lineas_posiciones)
bajas_clubes = extraer_lineas_baja(lineas_clubes)
bajas_edades = extraer_lineas_baja(lineas_edades)

bajas_posiciones, bajas_clubes, bajas_edades

(['Leonardo Balerdi - Defensor',
  'AFA: Lista de los 26 jugadores de la Seleccion Argentina para defender el titulo en la Copa del Mundo 2026, publicada el 28 de mayo de 2026.',
  "Reuters: Argentina's Balerdi out of World Cup with calf injury, publicado el 6 de junio de 2026."],
 ['Leonardo Balerdi - Marseille - Europa'],
 ['Leonardo Balerdi - 27'])

### Pregunta

> ¿Conviene eliminar estos datos, mantenerlos en otra tabla, o agregarlos con una columna `estado`?

## Desafío

Modificar el pipeline para que el `DataFrame` final incluya también a los jugadores dados de baja, agregando una columna:

```text
estado
```

con valores posibles:

```text
convocado
baja_pendiente_reemplazo
```

Después responder:

- ¿Cuántos jugadores hay por `estado`?
- ¿La baja modifica el promedio de edad?
- ¿Qué análisis cambia si incluimos o excluimos las bajas?

## Cierre

Este ejercicio muestra una situación común en ETL: los datos existen, pero no llegan directamente como una tabla limpia.

El trabajo consiste en reconocer estructura dentro de texto libre, separar datos de metadatos, validar consistencia entre fuentes, integrar tablas por una clave y recién después analizar.